In [373]:
import pandas as pd
import altair as alt
import panel as pn
from panel.interact import interact
from vega_datasets import data

pn.extension('vega')
alt.renderers.enable("browser")
imd_df = pd.read_csv('country_economics_data.csv')
interval = alt.selection_interval()
region_options = sorted(imd_df["Region"].unique())
selection = alt.selection_point(fields=['Region'], name='Region')


region_dropdown = alt.binding_select(options=region_options, name="Region")
region_select = alt.selection_point(fields=['Region'], bind=region_dropdown)
def region(base):
    filter_region = base.add_params(
        region_select
    ).transform_filter(
        region_select
    )
    return filter_region

In [374]:
def top_gdp():
    chart = (
        alt.Chart(imd_df)
        # 1️⃣ Filter data by region (from your selection)
        .transform_filter(selection)
        # 2️⃣ Rank GDP *within* the filtered dataset
        .transform_window(
            rank='rank(GDP)',
            sort=[alt.SortField('GDP', order='descending')],
            groupby=['Region']  # 🧠 ensures ranking restarts per region
        )
        # 3️⃣ Keep only top 10 per region
        .transform_filter('datum.rank <= 10')
        .mark_bar()
        .encode(
            x=alt.X('GDP:Q', title='GDP (in billions)'),
            y=alt.Y('Name:N', sort='-x', title='Country'),
            color=alt.Color('Region:N', legend=alt.Legend(orient="left")),
        )
        .properties(
            title="Top GDPs (updates by region)",
            width=480,
            height=400
        )
    )
    return chart


In [375]:
def world_map():
    imd_df['Name'] = imd_df['Name'].str.strip()

    countries = alt.topo_feature(data.world_110m.url, 'countries')

    variable = 'Population'  

    world_map = alt.Chart(countries).mark_geoshape(
        stroke='white',
        strokeWidth=0.5
    ).encode(
        color=alt.Color(
            f"{variable}:Q",
            title=f"{variable} (Billion USD)",
            legend=alt.Legend(
                orient="bottom",
                gradientLength=400,
                gradientThickness=30
            )
        )

        tooltip=[
            alt.Tooltip('Name:N', title='Country'),
            alt.Tooltip('Population:Q', title='Population (Millions)'),
            alt.Tooltip('Region:N'),
            alt.Tooltip('Capital:N')
        ]
    ).transform_lookup(
        lookup='id',
        from_=alt.LookupData(imd_df, 'ID', ['Name', 'GDP', 'Population', 'Region', 'Capital'])
    ).project(
        type='mercator'
    ).properties(
        width=960,
        height=650,
        title='Population by Country'
    )
    return world_map


SyntaxError: invalid syntax. Perhaps you forgot a comma? (148053797.py, line 12)

In [ ]:
import altair as alt

# define global selection once
interval = alt.selection_interval()

def gdp_popu():
    return (
        alt.Chart(imd_df)
        .mark_point()
        .encode(
            x=alt.X('GDP:Q', title='GDP (in billions)'),
            y=alt.Y('Population:Q', title='Population (millions)'),
            color=alt.Color('Region:N'),
        )
        .transform_filter(selection)
        .properties(title="GDP vs Population",
                    width = 640,
                    height = 270)
    )

def int_inf():
    return (
        alt.Chart(imd_df)
        .mark_point()
        .encode(
            x=alt.X('Interest Rate:Q', title='Interest'),
            y=alt.Y('Inflation Rate:Q', title='Inflation'),
            color=alt.Color('Region:N'),
        )
        .transform_filter(selection)
        .properties(title="Interest vs Inflation",
                    width = 640,
                    height = 270)
    )

def GDP_grow_inf():
    return (
        alt.Chart(imd_df)
        .mark_point()
        .encode(
            x=alt.X('Inflation Rate:Q', title='GDP Growth'),
            y=alt.Y('GDP Growth:Q', title='Inflation'),
            color=alt.Color('Region:N', legend=alt.Legend(orient="bottom")),
        )
        .transform_filter(selection)
        .properties(title="GDP Growth vs Inflation",
                    width = 640,
                    height = 270)
    )


def line_of_bf(base, x, y):
    regression_line = (
        base.transform_regression(x, y)
        .mark_line(color='red')
        .encode(color=alt.value('red'))
    )
    return (base + regression_line).add_selection(selection)


In [ ]:
def unemployment_region():
    return (
        alt.Chart(imd_df)
        .mark_boxplot(extent='min-max', color="#4C78A8")
        .encode(
            x=alt.X("Region:N", title="Region", sort="-y"),
            y=alt.Y("Jobless Rate:Q", title="Jobless Rate (%)"),
            tooltip=["Region", "Jobless Rate"]
        )
        .properties(
            title="Unemployment Rate by Region",
            width=480,
            height=540
        )
    )


In [ ]:
import altair as alt

imd_df["Fiscal Status"] = imd_df["Gov. Budget"].apply(lambda x: "Surplus" if x > 0 else "Deficit")

chart = (
    alt.Chart(imd_df)
    .mark_circle(size=90, opacity=0.75)
    .encode(
        x=alt.X("Debt/GDP:Q", title="Debt to GDP (%)"),
        y=alt.Y("Gov. Budget:Q", title="Budget Balance (% of GDP)"),
        color=alt.Color("Fiscal Status:N", scale=alt.Scale(domain=["Surplus", "Deficit"], range=["#2ca02c", "#d62728"])),
        shape=alt.Shape("Region:N", title="Region"),
        tooltip=["Region:N", "Debt/GDP:Q", "Gov. Budget:Q"]
    )
    .properties(
        title="Fiscal Balance vs Debt Load",
        width=700,
        height=450
    )
)


In [ ]:
world_by_pop = world_map().add_params(selection)
top_10_gdps = region(top_gdp())
unemployment_region = unemployment_region()

chart1 = line_of_bf(int_inf(), 'Interest Rate', 'Inflation Rate')
chart2 = line_of_bf(gdp_popu(), 'GDP', 'Population')
chart3 = line_of_bf(GDP_grow_inf(), 'Inflation Rate', 'GDP Growth')


horizontal = alt.hconcat(top_10_gdps, world_by_pop, unemployment_region)
horizontal2 = region(alt.hconcat(chart1, chart2, chart3))
final_output = alt.vconcat(horizontal, horizontal2)

final_output

C:\Users\44748\AppData\Local\Temp\ipykernel_5260\1275648800.py:58: AltairDeprecationWarning: 
Deprecated since `altair=5.0.0`. Use add_params instead.
  return (base + regression_line).add_selection(selection)


TypeError: Objects with 'config' attribute cannot be used within HConcatChart. Consider defining the config attribute in the HConcatChart object instead.